Here's your plan:

- input → tokens
- token embedding + positional embedding (add them)
- one-head attention ✓ (done)
- multi-head attention — heads in parallel, concat, + an output projection back to n_embd
- feedforward (Linear → ReLU → Linear, 4× wide) ← was missing
- assemble a Block = LN→MHA→+residual, LN→FFN→+residual
- stack N blocks (depth)
- final LayerNorm + lm_head (Linear n_embd → vocab_size to get logits) ← easy to forget
- forward → cross-entropy loss (reshape logits to (B*T, vocab), targets to (B*T,))
- backward + optimizer step (zero_grad → backward → step, AdamW) — this is your "training loop"
- sampling / generate

In [100]:
import torch 
import torch.nn.functional as F
import torch.nn as nn 

In [101]:
with open('./input.txt','r') as f:
    x = f.read()

In [102]:
T = 4
B = 10
C = 4
heads = 2

In [103]:
inp = torch.randint(1,len(x)-T,(B,))
inp = [x[i.item():i.item()+T] for i in inp]
stoi = {}
itos = {}
for i,s in enumerate(sorted(list(set(x)))):
    stoi[s] = i 
    itos[i] = s 


inp2 = []
for i in inp:
    inp2.append([stoi[j] for j in i])

inp2 = torch.tensor(inp2)
inp2

tensor([[40, 56, 43, 39],
        [ 1, 40, 39, 61],
        [46, 39, 50, 50],
        [59, 56, 42, 43],
        [52, 53, 58,  1],
        [58,  1, 51, 39],
        [ 1, 58, 46, 43],
        [57,  1, 52, 53],
        [42,  1, 44, 50],
        [50, 50,  5, 42]])

cleaner implementaion 

In [192]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        assert C%heads == 0 , "C must be divisible by heads"
        self.head_size = C//heads
        self.embd = nn.Embedding(len(stoi),C)
        self.posi = nn.Embedding(T,C)
        self.key = nn.ModuleList([nn.Linear(C,self.head_size,bias=False) for _ in range(heads)])
        self.query = nn.ModuleList([nn.Linear(C,self.head_size,bias=False) for _ in range(heads)])
        self.value = nn.ModuleList([nn.Linear(C,self.head_size,bias=False) for _ in range(heads)])
        

    def nn_embed(self,x):
        embedings = self.embd(x)
        embedings = embedings+self.posi(torch.arange(T))
        return embedings

    def attention(self,x):
        out = None
        for i in range(heads):
            query = self.query[i](x)
            key = self.key[i](x)
            value = self.value[i](x)
            
            qk = query@key.transpose(-2,-1) * self.head_size**-0.5
            tril = torch.tril(torch.ones(T,T))
            we = qk.masked_fill(tril == 0, float('-inf'))
            we = torch.softmax(we,dim=-1)
            temp = we@value
            if out is None:
                out = temp
            else:
                out = torch.cat([out,temp],dim=-1)
        return out

        

In [193]:
gpt = GPT()
emb = gpt.nn_embed(inp2)
att = gpt.attention(emb)
att = emb+att

In [194]:
att

tensor([[[ 1.7033,  1.2321,  1.6832,  1.5301],
         [-0.2308,  0.6616, -0.7909,  1.3181],
         [ 0.3815,  1.3748, -1.5641, -0.0637],
         [ 0.8987, -0.4697, -1.3775, -0.1461]],

        [[ 0.0650,  0.7036,  0.3217,  0.3303],
         [ 1.7551,  0.7776,  1.4452,  1.9074],
         [ 1.9929, -0.0959, -0.2918,  0.6028],
         [ 0.9163,  1.0747, -1.8611, -0.0519]],

        [[ 0.9452, -0.9089,  0.8733, -0.3721],
         [ 1.8513, -1.4592,  1.5502, -0.6597],
         [ 1.5099, -0.6926,  0.8166,  1.5087],
         [ 0.4257, -0.9015, -0.6375,  0.8334]],

        [[-0.8300,  1.2501,  0.4048, -0.3425],
         [-0.0182,  0.4033, -0.8016,  1.5276],
         [ 0.6510,  1.7702, -1.6227,  2.9470],
         [-0.7584,  1.0210, -2.6048, -1.0149]],

        [[ 1.6926, -2.3977,  2.0192, -1.2306],
         [ 2.2730, -0.3823,  1.2149,  1.5657],
         [ 0.5684,  0.1593, -0.2601,  1.9267],
         [-0.4089,  0.7425, -1.7946,  1.1501]],

        [[ 0.3873, -0.4254,  0.5038,  0.3476],
   

In [190]:
for name, p in gpt.named_parameters():
    print(name, tuple(p.shape))


embd.weight (65, 4)
posi.weight (4, 4)
key.0.weight (2, 4)
key.1.weight (2, 4)
query.0.weight (2, 4)
query.1.weight (2, 4)
value.0.weight (2, 4)
value.1.weight (2, 4)
